# Pilot Analysis: Game Evidence Graph

This notebook is for asking early research/design questions against the generated datasets. Start with deterministic table queries. Use the optional LLM synthesis section only when you want prose answers over selected evidence rows.

## 1. Setup

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 220)

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src/game_evidence_graph").exists():
            return candidate
    raise RuntimeError("Could not find project root. Launch Jupyter from the repo or notebooks directory.")

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data" / "processed"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)

PROJECT_ROOT: /Users/srishtisaha/Desktop/infinite_lives/game-evidence-graph
DATA_DIR: /Users/srishtisaha/Desktop/infinite_lives/game-evidence-graph/data/processed


## 2. Load Data

In [2]:
df = pd.read_csv(DATA_DIR / "mechanic_intervention_dataset.csv")
edges = pd.read_csv(DATA_DIR / "evidence_edge_table.csv")
study_ontology = json.loads((DATA_DIR / "study_ontology.json").read_text())
game_ontology = json.loads((DATA_DIR / "game_ontology.json").read_text())

print("Dataset C rows:", len(df))
print("Supported evidence rows:", int(df["is_supported_evidence"].astype(bool).sum()))
print("Evidence graph edges:", len(edges))
print("Games:", len(game_ontology.get("games", [])))

Dataset C rows: 5159
Supported evidence rows: 3964
Evidence graph edges: 3964
Games: 381


## 3. Core Query Helpers

In [3]:
ANSWER_COLS = [
    "paper_id",
    "paper_title",
    "year",
    "game_name",
    "mechanic_name",
    "population_raw",
    "study_design",
    "outcome_canonical",
    "outcome_category",
    "effect_direction",
    "effect_size_raw",
    "p_value",
    "evidence_strength",
    "edge_weight",
    "source_quote",
    "review_status",
]

def supported(data: pd.DataFrame = df) -> pd.DataFrame:
    return data[data["is_supported_evidence"].astype(bool)].copy()

def contains(series: pd.Series, term: str) -> pd.Series:
    return series.astype(str).str.contains(term, case=False, na=False, regex=False)

def search_evidence(
    term: str | None = None,
    *,
    game: str | None = None,
    mechanic: str | None = None,
    outcome: str | None = None,
    category: str | None = None,
    effect: str | None = None,
    require_supported: bool = True,
    n: int = 25,
) -> pd.DataFrame:
    data = supported(df) if require_supported else df.copy()
    mask = pd.Series(True, index=data.index)

    if term:
        term_mask = pd.Series(False, index=data.index)
        for col in ["paper_title", "game_name", "mechanic_name", "outcome_raw", "outcome_canonical", "outcome_category", "source_quote"]:
            if col in data:
                term_mask = term_mask | contains(data[col], term)
        mask &= term_mask
    if game:
        mask &= contains(data["game_name"], game)
    if mechanic:
        mask &= contains(data["mechanic_name"], mechanic)
    if outcome:
        mask &= contains(data["outcome_canonical"], outcome) | contains(data["outcome_raw"], outcome)
    if category:
        mask &= contains(data["outcome_category"], category)
    if effect:
        mask &= data["effect_direction"].astype(str).str.lower().eq(effect.lower())

    result = data.loc[mask, [col for col in ANSWER_COLS if col in data.columns]].copy()
    if "edge_weight" in result:
        result = result.sort_values("edge_weight", ascending=False)
    return result.head(n)

def summarize_counts(data: pd.DataFrame) -> pd.DataFrame:
    if data.empty:
        return pd.DataFrame()
    return pd.DataFrame(
        {
            "rows": [len(data)],
            "papers": [data["paper_id"].nunique()],
            "games": [data["game_name"].nunique(dropna=True)],
            "mechanics": [data["mechanic_name"].nunique(dropna=True)],
            "outcomes": [data["outcome_canonical"].nunique(dropna=True)],
            "positive": [(data["effect_direction"] == "positive").sum()],
            "negative": [(data["effect_direction"] == "negative").sum()],
            "mixed": [(data["effect_direction"] == "mixed").sum()],
            "null": [(data["effect_direction"] == "null").sum()],
        }
    )

## 4. Candidate Pilot Questions

These are good first-pass questions because they map cleanly onto the structured fields.

In [4]:
pilot_questions = [
    "Which game mechanics have positive evidence for neuro/cognitive outcomes?",
    "What evidence links movement-control or gesture-input games to physiological outcomes?",
    "What evidence exists for Minecraft or cooperative play improving social outcomes?",
    "What evidence links first-person shooter/action video games to aggression or affective outcomes?",
    "Which named games appear most often in supported evidence rows?",
    "Which supported claims have the strongest edge weights?",
    "Where do we have null-game rows, and are those correctly generic studies?",
]

for i, question in enumerate(pilot_questions, start=1):
    print(f"{i}. {question}")

1. Which game mechanics have positive evidence for neuro/cognitive outcomes?
2. What evidence links movement-control or gesture-input games to physiological outcomes?
3. What evidence exists for Minecraft or cooperative play improving social outcomes?
4. What evidence links first-person shooter/action video games to aggression or affective outcomes?
5. Which named games appear most often in supported evidence rows?
6. Which supported claims have the strongest edge weights?
7. Where do we have null-game rows, and are those correctly generic studies?


## 5. Query 1: Mechanics With Positive Neuro/Cognitive Evidence

In [5]:
q1 = search_evidence(category="Neuro", effect="positive", n=50)
display(summarize_counts(q1))
display(q1)

,rows,papers,games,mechanics,outcomes,positive,negative,mixed,null
0,50,9,17,8,14,50,0,0,0


,paper_id,paper_title,year,game_name,mechanic_name,population_raw,study_design,outcome_canonical,outcome_category,effect_direction,effect_size_raw,p_value,evidence_strength,edge_weight,source_quote,review_status
5083,paper_302,Short‑Term Effects of Video‑Games on Cognitive Enhancement:,2021,fighting video-game,not_reported,healthy young adults,randomized_controlled_trial,single word reading and pseudoword reading improvement associated with play-driven biochemical activation,Neuro/Cognitive,positive,not_reported,not_reported,randomized_controlled_trial,0.2250,play-driven biochemical activation boosted single word and pseudoword reading.,needs_review
3301,paper_191,This is an open access article under the CC BY-NC-ND license (https://creativecommons.org/licenses/by-nc-nd/4.0) Peer-review under responsibility of the scientific committee of the 5th International Conference on Com...,2021,not_reported,not_reported,"Participants were generally people who don’t play video games naturally, split into 2 random groups.",randomized_controlled_trial,improvement in target cognitive tasks,neurocognitive,positive,not_reported,not_reported,randomized_controlled_trial,0.2250,"An experiment conducted with pre-testing and post-testing. Participants were generally people who don’t play video games naturally, split into 2 random groups. One group play action video game for 10 – 50 hours, with...",needs_review
4039,paper_257,Open Access The effects of video game therapy on balance and attention in chronic ambulatory traumatic brain injury: an exploratory study,2013,Kinect Sports,Movement Control,We enrolled chronic traumatic brain injury patients (n = 21) that randomly received VGT or BPT for 3 sessions per week for 6 weeks.,randomized_controlled_trial,Selective visual attention (Go/No-go task),Attention / Neurocognitive,positive,not_reported,<0.01,randomized_controlled_trial,0.2250,Selective attention improved significantly in the VGT group (p < 0.01).,needs_review
4038,paper_257,Open Access The effects of video game therapy on balance and attention in chronic ambulatory traumatic brain injury: an exploratory study,2013,Kinect Sports,Sports Simulation,We enrolled chronic traumatic brain injury patients (n = 21) that randomly received VGT or BPT for 3 sessions per week for 6 weeks.,randomized_controlled_trial,Selective visual attention (Go/No-go task),Attention / Neurocognitive,positive,not_reported,<0.01,randomized_controlled_trial,0.2250,Selective attention improved significantly in the VGT group (p < 0.01).,needs_review
4037,paper_257,Open Access The effects of video game therapy on balance and attention in chronic ambulatory traumatic brain injury: an exploratory study,2013,Kinect Sports,Gesture Input,We enrolled chronic traumatic brain injury patients (n = 21) that randomly received VGT or BPT for 3 sessions per week for 6 weeks.,randomized_controlled_trial,Selective visual attention (Go/No-go task),Attention / Neurocognitive,positive,not_reported,<0.01,randomized_controlled_trial,0.2250,Selective attention improved significantly in the VGT group (p < 0.01).,needs_review
4024,paper_257,Open Access The effects of video game therapy on balance and attention in chronic ambulatory traumatic brain injury: an exploratory study,2013,Kinect Adventures,Movement Control,We enrolled chronic traumatic brain injury patients (n = 21) that randomly received VGT or BPT for 3 sessions per week for 6 weeks.,randomized_controlled_trial,Selective visual attention (Go/No-go task),Attention / Neurocognitive,positive,not_reported,<0.01,randomized_controlled_trial,0.2250,Selective attention improved significantly in the VGT group (p < 0.01).,needs_review
4023,paper_257,Open Access The effects of video game therapy on balance and attention in chronic ambulatory traumatic brain injury: an exploratory study,2013,Kinect Adventures,Sports Simulation,We enrolled chronic traumatic brain injury patients (n = 21) that randomly received VGT or BPT for 3 sessions per week for 

In [6]:
display(
    q1.groupby(["mechanic_name", "outcome_canonical", "effect_direction"], dropna=False)
    .agg(rows=("paper_id", "size"), papers=("paper_id", "nunique"), mean_edge_weight=("edge_weight", "mean"))
    .sort_values(["papers", "rows", "mean_edge_weight"], ascending=False)
    .head(30)
)

rows  \
mechanic_name     outcome_canonical                                                                                         effect_direction         
Gesture Input     Selective visual attention (Go/No-go task)                                                                positive             3   
Movement Control  Selective visual attention (Go/No-go task)                                                                positive             3   
Sports Simulation Selective visual attention (Go/No-go task)                                                                positive             3   
not_reported      hippocampal-based memory (interpreted improvement after intervention)                                     positive             3   
                  lure discrimination index (LDI) from the Mnemonic Similarity Task (hippocampal-based memory)              positive             3   
                  contextual reading (reading in context) improvement correlated with changes in positive emotions          positive             2   
                  improvement in target cognitive tasks                                                                     positive             2   
                  single word reading and pseudoword reading improvement associated with play-driven biochemical activation positive             2   
Combat            visual attention / visual attention span                                                                  positive             2   
Movement Control  visual attention / visual attention span                                                                  positive             2   
Rapid Response    visual attention / visual attention span                                                                  positive             2   
Targeting         visual attention / visual attention span                                                                  positive             2   
not_reported      declarative memory performance                                                                            positive             2   
                  EEG theta band power (frontal regions)                                                                    positive             2   
                  P2 component latency (occipito-temporal)                                                                  positive             2   
Puzzle Solving    sensorimotor performance / reduction in sensorimotor disorder measures                                    positive             1   
                  visual perception (breadth/global)                                                                        positive             1   
Targeting         sensorimotor performance / reduction in sensorimotor disorder measures                                    positive             1   
                  visual perception (breadth/global)                                                                        positive             1   
not_reported      Selective visual attention (Go/No-go task)                                                                positive             1   
                  sensorimotor performance / reduction in sensorimotor disorder measures                                    positive             1   
                  visual perception (breadth/global)                                                                        positive             1   
Puzzle Solving    hippocampal-based memory (interpreted improvement after intervention)                                     positive             1   
                  lure discrimination index (LDI) from the Mnemonic Similarity Task (hippocampal-based memory)              positive             1   
Targeting         hippocampal-based memory (interpreted improvement after intervention)                                     positive             1   
                  lure discrimination index (LDI) from the Mnemonic Similarity Task (hippoca

## 6. Query 2: Movement/Gesture Evidence for Physiological Outcomes

In [7]:
movement = supported(df)
movement = movement[
    movement["mechanic_name"].astype(str).str.contains("Movement Control|Gesture Input|Balance Training|Physical Imitation", case=False, na=False, regex=True)
    & movement["outcome_category"].astype(str).str.contains("Physiological|Anthropometric|Balance|Motor", case=False, na=False, regex=True)
]
movement = movement[[col for col in ANSWER_COLS if col in movement.columns]].sort_values("edge_weight", ascending=False)

display(summarize_counts(movement))
display(movement.head(50))

,rows,papers,games,mechanics,outcomes,positive,negative,mixed,null
0,307,13,25,4,29,111,6,18,0


,paper_id,paper_title,year,game_name,mechanic_name,population_raw,study_design,outcome_canonical,outcome_category,effect_direction,effect_size_raw,p_value,evidence_strength,edge_weight,source_quote,review_status
3998,paper_257,Open Access The effects of video game therapy on balance and attention in chronic ambulatory traumatic brain injury: an exploratory study,2013,X-Box 360 Kinect,Gesture Input,We enrolled chronic traumatic brain injury patients (n = 21) that randomly received VGT or BPT for 3 sessions per week for 6 weeks.,randomized_controlled_trial,Unified Balance Scale (UBS),Balance/Mobility,positive,not_reported,<0.05,randomized_controlled_trial,0.2250,"Both groups improved in CB&M scores, but only the VGT group increased on the UBS and TUG with a between-group significance (p < 0.05).",needs_review
4030,paper_257,Open Access The effects of video game therapy on balance and attention in chronic ambulatory traumatic brain injury: an exploratory study,2013,Kinect Sports,Movement Control,We enrolled chronic traumatic brain injury patients (n = 21) that randomly received VGT or BPT for 3 sessions per week for 6 weeks.,randomized_controlled_trial,Unified Balance Scale (UBS),Balance/Mobility,positive,not_reported,<0.05,randomized_controlled_trial,0.2250,"Both groups improved in CB&M scores, but only the VGT group increased on the UBS and TUG with a between-group significance (p < 0.05).",needs_review
4028,paper_257,Open Access The effects of video game therapy on balance and attention in chronic ambulatory traumatic brain injury: an exploratory study,2013,Kinect Sports,Gesture Input,We enrolled chronic traumatic brain injury patients (n = 21) that randomly received VGT or BPT for 3 sessions per week for 6 weeks.,randomized_controlled_trial,Unified Balance Scale (UBS),Balance/Mobility,positive,not_reported,<0.05,randomized_controlled_trial,0.2250,"Both groups improved in CB&M scores, but only the VGT group increased on the UBS and TUG with a between-group significance (p < 0.05).",needs_review
4027,paper_257,Open Access The effects of video game therapy on balance and attention in chronic ambulatory traumatic brain injury: an exploratory study,2013,Kinect Sports,Movement Control,We enrolled chronic traumatic brain injury patients (n = 21) that randomly received VGT or BPT for 3 sessions per week for 6 weeks.,randomized_controlled_trial,Community Balance and Mobility Scale (CB&M),Balance/Mobility,positive,not_reported,not_reported,randomized_controlled_trial,0.2250,"Both groups improved in CB&M scores, but only the VGT group increased on the UBS and TUG with a between-group significance (p < 0.05). Selective attention improved significantly in the VGT group (p < 0.01).",needs_review
4025,paper_257,Open Access The effects of video game therapy on balance and attention in chronic ambulatory traumatic brain injury: an exploratory study,2013,Kinect Sports,Gesture Input,We enrolled chronic traumatic brain injury patients (n = 21) that randomly received VGT or BPT for 3 sessions per week for 6 weeks.,randomized_controlled_trial,Community Balance and Mobility Scale (CB&M),Balance/Mobility,positive,not_reported,not_reported,randomized_controlled_trial,0.2250,"Both groups improved in CB&M scores, but only the VGT group increased on the UBS and TUG with a between-group significance (p < 0.05). Selective attention improved significantly in the VGT group (p < 0.01).",needs_review
4015,paper_257,Open Access The effects of video game therapy on balance and attention in chronic ambulatory traumatic brain injury: an exploratory study,2013,Kinect Adventures,Movement Control,We enrolled chronic traumatic brain injury patients (n = 21) that randomly received VGT or BPT for 3 sessions per week for 6 weeks.,randomized_controlled_trial,Unified Balance Scale (UBS),Balance/Mobility,positive,not_reported,<0.05,randomized_controlled_trial,0.2250,"Both groups improved in CB&M scores, but only the VGT group increased on the UBS and TUG with a between-gro

## 7. Query 3: Minecraft / Cooperative Play / Social Outcomes

In [8]:
social = supported(df)
social = social[
    (
        social["game_name"].astype(str).str.contains("Minecraft", case=False, na=False)
        | social["mechanic_name"].astype(str).str.contains("Cooperative|Cooperation|Perspective", case=False, na=False, regex=True)
    )
    & social[["outcome_category", "outcome_canonical", "outcome_raw"]]
    .astype(str)
    .agg(" ".join, axis=1)
    .str.contains("Social|intergroup|tolerance|prosocial|empathy|cooperation", case=False, na=False, regex=True)
]
social = social[[col for col in ANSWER_COLS if col in social.columns]].sort_values("edge_weight", ascending=False)

display(summarize_counts(social))
display(social.head(50))

,rows,papers,games,mechanics,outcomes,positive,negative,mixed,null
0,10,2,2,5,3,10,0,0,0


,paper_id,paper_title,year,game_name,mechanic_name,population_raw,study_design,outcome_canonical,outcome_category,effect_direction,effect_size_raw,p_value,evidence_strength,edge_weight,source_quote,review_status
1945,paper_096,Underlying factors of social capital acquisition in the context of online-gaming: Comparing World of Warcraft and Counter-Strike,2014,Counter-Strike,Cooperation,A sample of 682 clan/guild players of the games Counter-Strike and World of Warcraft was recruited via Internet and was asked to fill out a questionnaire.,correlational,bridging social capital,social_capital,positive,NaN,NaN,correlational,0.0473,"Results of a path analysis support our assumptions and enhance previous ﬁndings that players of online-games especially gain positive social outcomes, when they go beyond the game and join game-related groups, engage...",needs_review
1956,paper_096,Underlying factors of social capital acquisition in the context of online-gaming: Comparing World of Warcraft and Counter-Strike,2014,Counter-Strike,Cooperation,A sample of 682 clan/guild players of the games Counter-Strike and World of Warcraft was recruited via Internet and was asked to fill out a questionnaire.,correlational,bonding social capital,social_capital,positive,NaN,NaN,correlational,0.0473,"we hypothesize that ... foster self-disclosure towards fellow players, which together facilitates the formation of bridging and bonding social capital.",needs_review
488,paper_011,Gaming for peace: Virtual contact through cooperative video gaming increases children's intergroup tolerance in the context of the Israeli–Palestinian conflict,2018,Minecraft,Cooperative Play,mixed virtual teams of Jewish-Israeli and Palestinian-Israeli children,not_reported,willingness to engage in social contact (behavioral_intent),behavioral_intent,positive,NaN,NaN,not_reported,0.0135,"children who participated in the intergroup program showed reduced intergroup bias on both cognitive and emotional indicators, including reduced stereotypical views, negative emotions and discriminatory tendencies to...",needs_review
489,paper_011,Gaming for peace: Virtual contact through cooperative video gaming increases children's intergroup tolerance in the context of the Israeli–Palestinian conflict,2018,Minecraft,Building,mixed virtual teams of Jewish-Israeli and Palestinian-Israeli children,not_reported,willingness to engage in social contact (behavioral_intent),behavioral_intent,positive,NaN,NaN,not_reported,0.0135,"children who participated in the intergroup program showed reduced intergroup bias on both cognitive and emotional indicators, including reduced stereotypical views, negative emotions and discriminatory tendencies to...",needs_review
490,paper_011,Gaming for peace: Virtual contact through cooperative video gaming increases children's intergroup tolerance in the context of the Israeli–Palestinian conflict,2018,Minecraft,Exploration,mixed virtual teams of Jewish-Israeli and Palestinian-Israeli children,not_reported,willingness to engage in social contact (behavioral_intent),behavioral_intent,positive,NaN,NaN,not_reported,0.0135,"children who participated in the intergroup program showed reduced intergroup bias on both cognitive and emotional indicators, including reduced stereotypical views, negative emotions and discriminatory tendencies to...",needs_review
491,paper_011,Gaming for peace: Virtual contact through cooperative video gaming increases children's intergroup tolerance in the context of the Israeli–Palestinian conflict,2018,Minecraft,Perspective Taking,mixed virtual teams of Jewish-Israeli and Palestinian-Israeli children,not_reported,willingness to engage in social contact (behavioral_intent),behavioral_intent,positive,NaN,NaN,not_reported,0.0135,"children who participated in the intergroup program showed reduced intergroup bias on both cognitive and emotional indicators, including reduced stereotypical views, negative emotions and discriminatory tendencies to...",needs_r

## 8. Query 4: FPS / Action Games and Aggression or Affective Outcomes

In [9]:
fps = supported(df)
fps = fps[
    fps[["game_name", "paper_title", "mechanic_name"]]
    .astype(str)
    .agg(" ".join, axis=1)
    .str.contains("first-person|shooter|FPS|action video|Counter-Strike|violent", case=False, na=False, regex=True)
    & fps[["outcome_category", "outcome_canonical", "outcome_raw"]]
    .astype(str)
    .agg(" ".join, axis=1)
    .str.contains("aggression|aggressive|affect|emotion|cortisol|arousal|attention|cognitive", case=False, na=False, regex=True)
]
fps = fps[[col for col in ANSWER_COLS if col in fps.columns]].sort_values("edge_weight", ascending=False)

display(summarize_counts(fps))
display(fps.head(50))

,rows,papers,games,mechanics,outcomes,positive,negative,mixed,null
0,402,27,36,25,66,312,40,21,0


,paper_id,paper_title,year,game_name,mechanic_name,population_raw,study_design,outcome_canonical,outcome_category,effect_direction,effect_size_raw,p_value,evidence_strength,edge_weight,source_quote,review_status
452,paper_009,Intense acts of violence during video game play make daily life aggression appear innocuous: A new mechanism why violent video games increase aggression,2012,Counterstrike,not_reported,"Participants were 82 adults (61 women, 21 men, mean age: 21.9 years).",randomized_controlled_trial,perceived aggressiveness,Psychological,negative,d = 0.84,< .001,randomized_controlled_trial,0.2250,"participants who played the violent video game perceived their daily life aggressive behavior as being less aggressive (M = 4.23, SD = 2.40; contrast weight: 3) compared to participants who played the violent video g...",needs_review
455,paper_009,Intense acts of violence during video game play make daily life aggression appear innocuous: A new mechanism why violent video games increase aggression,2012,Trooper Assassin,not_reported,"Participants were 82 adults (61 women, 21 men, mean age: 21.9 years).",randomized_controlled_trial,perceived aggressiveness,Psychological,negative,d = 0.84,< .001,randomized_controlled_trial,0.2250,"participants who played the violent video game perceived their daily life aggressive behavior as being less aggressive (M = 4.23, SD = 2.40; contrast weight: 3) compared to participants who played the violent video g...",needs_review
458,paper_009,Intense acts of violence during video game play make daily life aggression appear innocuous: A new mechanism why violent video games increase aggression,2012,Super Bubbles,not_reported,"Participants were 82 adults (61 women, 21 men, mean age: 21.9 years).",randomized_controlled_trial,perceived aggressiveness,Psychological,negative,d = 0.84,< .001,randomized_controlled_trial,0.2250,"participants who played the violent video game perceived their daily life aggressive behavior as being less aggressive (M = 4.23, SD = 2.40; contrast weight: 3) compared to participants who played the violent video g...",needs_review
461,paper_009,Intense acts of violence during video game play make daily life aggression appear innocuous: A new mechanism why violent video games increase aggression,2012,Penguin,not_reported,"Participants were 82 adults (61 women, 21 men, mean age: 21.9 years).",randomized_controlled_trial,perceived aggressiveness,Psychological,negative,d = 0.84,< .001,randomized_controlled_trial,0.2250,"participants who played the violent video game perceived their daily life aggressive behavior as being less aggressive (M = 4.23, SD = 2.40; contrast weight: 3) compared to participants who played the violent video g...",needs_review
4902,paper_292,"Developmental Dyslexia (Tn Wydell, Section Editor)",2015,action video game,Movement Control,children with DD,quasi_experimental,visual attention / visual attention span,Neuro/Cognitive,positive,not_reported,not_reported,quasi_experimental,0.1687,Franceschini et al. [100•] showed a direct effect of the AVG training on the spotlight of attention in children with DD. An improvement in the distribution and allocation of visuo-spatial attention was obtained in th...,needs_review
4904,paper_292,"Developmental Dyslexia (Tn Wydell, Section Editor)",2015,non-action video game,Rapid Response,children with DD,quasi_experimental,visual attention / visual attention span,Neuro/Cognitive,positive,not_reported,not_reported,quasi_experimental,0.1687,Franceschini et al. [100•] showed a direct effect of the AVG training on the spotlight of attention in children with DD. An improvement in the distribution and allocation of visuo-spatial attention was obtained in th...,needs_review
2614,paper_154,Enhancing Attentional Control: Lessons from Action Video Games,2019,NaN,not_reported,"players of a lab-designed video game that contained rewards outperformed individuals playing the same game without rewards on a declarative memory task (Prena et al.,

## 9. Query 5: Most Common Named Games in Supported Evidence

In [10]:
named_supported = supported(df)[supported(df)["game_id"].notna()].copy()
game_counts = (
    named_supported.groupby("game_name", dropna=False)
    .agg(rows=("paper_id", "size"), papers=("paper_id", "nunique"), outcomes=("outcome_canonical", "nunique"), mean_edge_weight=("edge_weight", "mean"))
    .sort_values(["papers", "rows"], ascending=False)
)
display(game_counts.head(50))

,rows,papers,outcomes,mean_edge_weight
game_name,,,,
not_reported,101,13,44,0.108370
World of Warcraft,42,9,31,0.077429
Nintendo Wii,40,8,23,0.134770
MMORPG,51,6,28,0.071663
video games,23,6,23,0.055191
Call of Duty,21,6,19,0.091724
action video games,280,5,17,0.093311
first-person shooter,140,5,19,0.062529
League of Legends,117,4,9,0.069767


## 10. Query 6: Strongest Supported Claims

In [11]:
display(supported(df)[[col for col in ANSWER_COLS if col in df.columns]].sort_values("edge_weight", ascending=False).head(50))

,paper_id,paper_title,year,game_name,mechanic_name,population_raw,study_design,outcome_canonical,outcome_category,effect_direction,effect_size_raw,p_value,evidence_strength,edge_weight,source_quote,review_status
4015,paper_257,Open Access The effects of video game therapy on balance and attention in chronic ambulatory traumatic brain injury: an exploratory study,2013,Kinect Adventures,Movement Control,We enrolled chronic traumatic brain injury patients (n = 21) that randomly received VGT or BPT for 3 sessions per week for 6 weeks.,randomized_controlled_trial,Unified Balance Scale (UBS),Balance/Mobility,positive,not_reported,<0.05,randomized_controlled_trial,0.225,"Both groups improved in CB&M scores, but only the VGT group increased on the UBS and TUG with a between-group significance (p < 0.05).",needs_review
3342,paper_196,Use of commercial video games to improve postural balance in patients with multiple sclerosis: A systematic review and meta-analysis of randomised controlled clinical trials,2021,NaN,not_reported,patients with multiple sclerosis (MS),randomized_controlled_trial,Berg Balance Scale (BBS),postural_balance,positive,DM: 5.30,< .001,randomized_controlled_trial,0.225,"We did observe intergroup differences in BBS scores in favour of video game therapy (DM: 5.30; 95% CI,3.39—7.21; P < .001; I2 = 0%), but these were not greater than the minimum detectable change reported in the liter...",needs_review
3340,paper_196,Use of commercial video games to improve postural balance in patients with multiple sclerosis: A systematic review and meta-analysis of randomised controlled clinical trials,2021,NaN,not_reported,patients with multiple sclerosis (MS),randomized_controlled_trial,Four Step Square Test,postural_balance,NaN,DM: −.74,.48,randomized_controlled_trial,0.225,"We found no signiﬁcant differences between the video game therapy group and the control group in Four Step Square Test (DM: −.74; 95% CI, −2.79 to 1.32; P = .48; I2 = 0%)",needs_review
3324,paper_194,"Impact of an active educational video game on children’s motivation, science knowledge, and physical activity",2014,not_reported,not_reported,"In this randomized controlled study, 53 elementary school students were assigned to an experimental condition or a comparison condition.",randomized_controlled_trial,situational interest (perceived),Psychological,positive,η2 = 0.301,p < 0.01,randomized_controlled_trial,0.225,"Students in the experimental condition perceived a higher level of situational interest than their counterparts in the comparison group (p < 0.01, and η2 = 0.301).",needs_review
3323,paper_194,"Impact of an active educational video game on children’s motivation, science knowledge, and physical activity",2014,not_reported,not_reported,"In this randomized controlled study, 53 elementary school students were assigned to an experimental condition or a comparison condition.",randomized_controlled_trial,physical activity intensity (heart rate),Physiological,positive,t = 7.212,p < 0.001,randomized_controlled_trial,0.225,"the students’ average heart rates (HRs) were in the Target-Heart-Rate-Zone (HR = 134 bpm), which was signiﬁcantly higher than the average HR (103 bpm) from the comparison condition (t = 7.212, p < 0.001).",needs_review
3322,paper_194,"Impact of an active educational video game on children’s motivation, science knowledge, and physical activity",2014,not_reported,not_reported,"In this randomized controlled study, 53 elementary school students were assigned to an experimental condition or a comparison condition.",randomized_controlled_trial,science knowledge (test performance),Learning,NaN,η2 = 0.486 (pre-post),p < 0.001 (pre-post); between-group difference not significant,randomized_controlled_trial,0.225,"students in both groups performed better on the post-test than they did on the pre-test (p < 0.001, η2 = 0.486), and their post-test scores did not differ signiﬁcantly.",needs_review
3321,paper_194,"Impact of an active educational video gam

## 11. Query 7: Null-Game Rows Audit

In [12]:
null_game = df[df["game_id"].isna()].copy()
display(summarize_counts(null_game))
display(null_game[[col for col in ANSWER_COLS if col in null_game.columns]].head(50))

,rows,papers,games,mechanics,outcomes,positive,negative,mixed,null
0,927,154,0,1,597,464,92,28,0


,paper_id,paper_title,year,game_name,mechanic_name,population_raw,study_design,outcome_canonical,outcome_category,effect_direction,effect_size_raw,p_value,evidence_strength,edge_weight,source_quote,review_status
302,paper_003,Interactive video games to reduce paediatric procedural pain and anxiety: a systematic review and meta-analysis,2020,NaN,not_reported,children (0e18 yr) undergoing painful procedures,not_reported,procedural pain,Pain,negative,SMD = e0.43,not_reported,not_reported,0.0225,Interactive video games appear to reduce children’s procedural pain (standardised mean difference [SMD]¼e0.43; 95% conﬁdence interval [CI]: e0.67 to e0.20),needs_review
303,paper_003,Interactive video games to reduce paediatric procedural pain and anxiety: a systematic review and meta-analysis,2020,NaN,not_reported,children (0e18 yr) undergoing painful procedures,not_reported,anxiety,Psychological,negative,SMD¼0.61,not_reported,not_reported,0.0225,"Interactive video games appear to reduce children’s procedural pain (standardised mean difference [SMD]¼e0.43; 95% conﬁdence interval [CI]: e0.67 to e0.20), anxiety (SMD¼0.61; 95% CI: e0.88 to e0.34), and caregivers’...",needs_review
304,paper_003,Interactive video games to reduce paediatric procedural pain and anxiety: a systematic review and meta-analysis,2020,NaN,not_reported,children (0e18 yr) undergoing painful procedures,not_reported,caregiver anxiety,Psychological,negative,SMD¼e0.31,not_reported,not_reported,0.0225,"Interactive video games appear to reduce children’s procedural pain (standardised mean difference [SMD]¼e0.43; 95% conﬁdence interval [CI]: e0.67 to e0.20), anxiety (SMD¼0.61; 95% CI: e0.88 to e0.34), and caregivers’...",needs_review
305,paper_003,Interactive video games to reduce paediatric procedural pain and anxiety: a systematic review and meta-analysis,2020,NaN,not_reported,children (0e18 yr) undergoing painful procedures,not_reported,procedure length,Physiological/Medical,NaN,not_reported,not_reported,not_reported,0.0225,"We also observed no difference between interactive video games compared with standard care for most medical outcomes (e.g. procedure length), except a reduced need for restraint.",needs_review
306,paper_003,Interactive video games to reduce paediatric procedural pain and anxiety: a systematic review and meta-analysis,2020,NaN,not_reported,children (0e18 yr) undergoing painful procedures,not_reported,need for physical restraint during procedure,Behavioral/Medical,negative,not_reported,not_reported,not_reported,0.0225,"We also observed no difference between interactive video games compared with standard care for most medical outcomes (e.g. procedure length), except a reduced need for restraint.",needs_review
307,paper_003,Interactive video games to reduce paediatric procedural pain and anxiety: a systematic review and meta-analysis,2020,NaN,not_reported,children (0e18 yr) undergoing painful procedures,not_reported,adverse effects,Safety,negative,minimal adverse effects reported,not_reported,not_reported,0.0225,Studies reported minimal adverse effects and typically had high intervention acceptability and satisfaction.,needs_review
308,paper_003,Interactive video games to reduce paediatric procedural pain and anxiety: a systematic review and meta-analysis,2020,NaN,not_reported,children (0e18 yr) undergoing painful procedures,not_reported,acceptability/satisfaction,Implementation/Acceptability,positive,typically high intervention acceptability and satisfaction,not_reported,not_reported,0.0225,Studies reported minimal adverse effects and typically had high intervention acceptability and satisfaction.,needs_review
333,paper_004,Exposure to ﬁrst-person shooter videogames is associated with multisensory temporal precision and migraine incidence,2020,NaN,not_reported,We tested this hypothesis by comparing the temporal proﬁle of the sound-induced illusion in a group of expert ﬁrst-person shooter gamers and a non-players group.,between_subject_observational,multise

## 12. A Simple Evidence Answer Template

This is not an AI agent. It gives a compact deterministic answer skeleton from selected rows.

In [13]:
def evidence_brief(question: str, rows: pd.DataFrame, max_quotes: int = 5) -> None:
    print("QUESTION:", question)
    print()
    if rows.empty:
        print("No matching supported evidence rows found.")
        return
    print("COUNTS")
    print("- rows:", len(rows))
    print("- papers:", rows["paper_id"].nunique())
    print("- games:", rows["game_name"].nunique(dropna=True))
    print("- mechanics:", rows["mechanic_name"].nunique(dropna=True))
    print()
    print("EFFECT DIRECTIONS")
    print(rows["effect_direction"].value_counts(dropna=False).to_string())
    print()
    print("TOP MECHANICS")
    print(rows["mechanic_name"].value_counts(dropna=False).head(10).to_string())
    print()
    print("REPRESENTATIVE QUOTES")
    quotes = rows.dropna(subset=["source_quote"]).sort_values("edge_weight", ascending=False)
    for _, row in quotes.head(max_quotes).iterrows():
        print(f"- {row['paper_id']} | {row.get('game_name')} | {row.get('mechanic_name')} -> {row.get('outcome_canonical')} [{row.get('effect_direction')}]")
        print(f"  {row['source_quote']}")

evidence_brief("What evidence links movement/gesture games to physiological outcomes?", movement)

QUESTION: What evidence links movement/gesture games to physiological outcomes?

COUNTS
- rows: 307
- papers: 13
- games: 25
- mechanics: 4

EFFECT DIRECTIONS
effect_direction
NaN         172
positive    111
mixed        18
negative      6

TOP MECHANICS
mechanic_name
Movement Control      138
Gesture Input         117
Physical Imitation     26
Balance Training       26

REPRESENTATIVE QUOTES
- paper_257 | X-Box 360 Kinect | Gesture Input -> Unified Balance Scale (UBS) [positive]
  Both groups improved in CB&M scores, but only the VGT group increased on the UBS and TUG with a between-group significance (p < 0.05).
- paper_257 | Kinect Adventures | Movement Control -> Community Balance and Mobility Scale (CB&M) [positive]
  Both groups improved in CB&M scores, but only the VGT group increased on the UBS and TUG with a between-group significance (p < 0.05). Selective attention improved significantly in the VGT group (p < 0.01).
- paper_257 | Kinect Sports | Movement Control -> Unified Ba

## 13. Optional LLM Synthesis

An AI agent is useful here if you want a prose answer that weighs rows, quotes caveats, and proposes follow-up filters. Keep the dataframe query deterministic first, then pass only the selected evidence rows to an LLM. This keeps the agent grounded.

In [14]:
# Optional. Requires OPENAI_API_KEY or the same key setup you used for the pipeline.
# This cell is intentionally not run by default.

def load_project_env(env_path: Path = PROJECT_ROOT / ".env") -> None:
    """Tiny .env loader so the notebook does not require python-dotenv."""
    if not env_path.exists():
        return
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ[key.strip()] = value.strip().strip("'\"")

def llm_synthesize(question: str, rows: pd.DataFrame, model: str | None = None, max_rows: int = 20) -> str:
    try:
        from openai import OpenAI
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "The notebook kernel does not have the openai package. Install it in this kernel, "
            "or switch the notebook kernel to the project .venv."
        ) from exc

    load_project_env()
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY was not found. Check the project .env or set it in this notebook kernel.")

    model = model or os.getenv("GAME_EVIDENCE_LLM_MODEL") or "gpt-4.1-mini"
    client = OpenAI(api_key=api_key)

    evidence_rows = rows.head(max_rows)[[col for col in ANSWER_COLS if col in rows.columns]].to_dict(orient="records")
    prompt = f"""
You are analyzing a literature-derived game evidence dataset.
Answer the question using only the evidence rows provided.
Be concise. Separate supported findings from caveats. Mention paper_id values.

Question: {question}

Evidence rows JSON:
{json.dumps(evidence_rows, indent=2)}
""".strip()

    response = client.responses.create(
        model=model,
        input=prompt,
    )
    return response.output_text

# Example usage after inspecting `movement` above:
# print(llm_synthesize("What evidence links movement/gesture games to physiological outcomes?", movement))

In [15]:
# First run this query cell or rerun it here
movement = supported(df)
movement = movement[
    movement["mechanic_name"].astype(str).str.contains(
        "Movement Control|Gesture Input|Balance Training|Physical Imitation",
        case=False,
        na=False,
        regex=True,
    )
    & movement["outcome_category"].astype(str).str.contains(
        "Physiological|Anthropometric|Balance|Motor",
        case=False,
        na=False,
        regex=True,
    )
]

movement = movement.sort_values("edge_weight", ascending=False)
display(movement.head(20))

,paper_id,paper_title,year,doi,study_id,intervention_id,condition_id,condition_type,game_id,game_name,game_match_confidence,mechanic_id,mechanic_name,mechanic_match_confidence,mechanic_set_id,co_mechanics,population_id,population_raw,sample_size,age_range,country,context,duration,study_design,treatment_raw,control_raw,outcome_id,outcome_raw,outcome_canonical,outcome_category,outcome_match_confidence,measurement_id,measurement_raw,measurement_type,measurement_match_confidence,effect_direction,effect_size_raw,effect_size_numeric,effect_metric,p_value,confidence_interval,effect_target_measure,evidence_strength,attribution_level,claim_type,claim_explicitness,source_quote,source_page,page_trace_status,extraction_confidence,is_supported_evidence,edge_weight,review_status,notes
3998,paper_257,Open Access The effects of video game therapy on balance and attention in chronic ambulatory traumatic brain injury: an exploratory study,2013,10.1186/s12883-017-0871-9,study_1,study_1_int_01,study_1_cond_treatment,treatment,g299,X-Box 360 Kinect,needs_review,m004,Gesture Input,high,ms_27805,Gesture Input|Sports Simulation|Movement Control,pop_001,We enrolled chronic traumatic brain injury patients (n = 21) that randomly received VGT or BPT for 3 sessions per week for 6 weeks.,21.0,NaN,NaN,"ambulatory chronic traumatic brain injury patients recruited from patients discharged at home (clinic database) or those receiving multidisciplinary inpatient rehabilitation at Ferrara University Hospital (Ferrara, I...",Each patient received three 1-h sessions per week over the course of 6 weeks.,randomized_controlled_trial,"Video game therapy was delivered with a video game console (X-Box 360 Kinect, Microsoft, Inc., Redmond, WA). Pre-selected games were chosen from “Kinect Adventures” and “Kinect Sports” that encompassed a wide range o...","Balance/rebalancing, postural stability and weight-shifting exercises with and without visual feedback were administered using a balance platform (Biodex Medical Systems, Inc., Shirley, NY). Each task was trained for...",study_1_out_002,Unified Balance Scale (UBS) score,Unified Balance Scale (UBS),Balance/Mobility,high,study_1_meas_002,Unified Balance Scale (UBS),clinical scale,high,positive,not_reported,NaN,NaN,<0.05,not_reported,NaN,randomized_controlled_trial,mechanic_set_inferred,reported_intervention_effect,inferred,"Both groups improved in CB&M scores, but only the VGT group increased on the UBS and TUG with a between-group significance (p < 0.05).",1.0,found,0.5,True,0.2250,needs_review,NaN
4030,paper_257,Open Access The effects of video game therapy on balance and attention in chronic ambulatory traumatic brain injury: an exploratory study,2013,10.1186/s12883-017-0871-9,study_1,study_1_int_01,study_1_cond_treatment,treatment,g11,Kinect Sports,needs_review,m002,Movement Control,high,ms_00302,Gesture Input|Sports Simulation|Movement Control,pop_001,We enrolled chronic traumatic brain injury patients (n = 21) that randomly received VGT or BPT for 3 sessions per week for 6 weeks.,21.0,NaN,NaN,"ambulatory chronic traumatic brain injury patients recruited from patients discharged at home (clinic database) or those receiving multidisciplinary inpatient rehabilitation at Ferrara University Hospital (Ferrara, I...",Each patient received three 1-h sessions per week over the course of 6 weeks.,randomized_controlled_trial,"Video game therapy was delivered with a video game console (X-Box 360 Kinect, Microsoft, Inc., Redmond, WA). Pre-selected games were chosen from “Kinect Adventures” and “Kinect Sports” that encompassed a wide range o...","Balance/rebalancing, postural stability and weight-shifting exercises with and without visual feedback were administered using a balance platform (Biodex Medical Systems, Inc., Shirley, NY). Each task was trained for...",study_1_out_002,Unified Balance Scale (UBS) score,Unified Balance Scale (UBS),Balance/Mobility,high,study_1_meas_002,Unified Balance Scale (UBS),clinical scale,high,posi

In [16]:
answer = llm_synthesize(
    "What evidence links movement/gesture games to physiological outcomes?",
    movement,
    max_rows=20,
)

print(answer)

Supported findings
- paper_170: In a quasi‑experimental study of 30 children with cystic fibrosis (age ~12 y) vs age/gender‑matched controls, active movement/gesture games (Wii Fit Free Jogging, Wii Sports Boxing; mechanics labelled Gesture Input, Movement Control, Physical Imitation, Balance Training) showed no significant difference in energy cost between groups ("No significant difference was seen between groups in the energy cost of playing active video games."). (paper_id: paper_170)

Caveats
- Study design was quasi‑experimental and sample size was small (n=30); effect sizes and p‑values are not reported in these rows.  
- The outcome assessed was energy cost only (one physiological measure) and was a between‑group comparison (CF vs controls), not necessarily a within‑subject physiological change from the games.  
- Other provided studies (paper_257) report balance/mobility and attention outcomes (positive), but do not report physiological outcomes in the supplied rows. (paper_id

In [17]:
behavioral = supported(df)

behavioral = behavioral[
    behavioral[["outcome_category", "outcome_canonical", "outcome_raw", "paper_title"]]
    .astype(str)
    .agg(" ".join, axis=1)
    .str.contains(
        "behavior|behaviour|aggression|aggressive|prosocial|cooperation|motivation|engagement|addiction|gaming disorder|social|purchase|intention",
        case=False,
        na=False,
        regex=True,
    )
]

behavioral = behavioral[
    [col for col in ANSWER_COLS if col in behavioral.columns]
].sort_values("edge_weight", ascending=False)

display(summarize_counts(behavioral))
display(behavioral.head(50))

,rows,papers,games,mechanics,outcomes,positive,negative,mixed,null
0,1413,136,187,36,372,1003,125,108,0


,paper_id,paper_title,year,game_name,mechanic_name,population_raw,study_design,outcome_canonical,outcome_category,effect_direction,effect_size_raw,p_value,evidence_strength,edge_weight,source_quote,review_status
460,paper_009,Intense acts of violence during video game play make daily life aggression appear innocuous: A new mechanism why violent video games increase aggression,2012,Super Bubbles,not_reported,"Participants were 82 adults (61 women, 21 men, mean age: 21.9 years).",randomized_controlled_trial,game liking,Psychological,negative,d = 1.48,< .001,randomized_controlled_trial,0.2250,"Liking of the violent video game (M = 2.32, SD = 1.27) was lower than liking of the neutral video game (M = 4.40, SD = 1.53), t(80) = 6.65, p b .001, d = 1.48.",needs_review
458,paper_009,Intense acts of violence during video game play make daily life aggression appear innocuous: A new mechanism why violent video games increase aggression,2012,Super Bubbles,not_reported,"Participants were 82 adults (61 women, 21 men, mean age: 21.9 years).",randomized_controlled_trial,perceived aggressiveness,Psychological,negative,d = 0.84,< .001,randomized_controlled_trial,0.2250,"participants who played the violent video game perceived their daily life aggressive behavior as being less aggressive (M = 4.23, SD = 2.40; contrast weight: 3) compared to participants who played the violent video g...",needs_review
3320,paper_194,"Impact of an active educational video game on children’s motivation, science knowledge, and physical activity",2014,not_reported,not_reported,"In this randomized controlled study, 53 elementary school students were assigned to an experimental condition or a comparison condition.",randomized_controlled_trial,physical activity intensity (heart rate),Physiological,positive,t = 7.212,p < 0.001,randomized_controlled_trial,0.2250,"the students’ average heart rates (HRs) were in the Target-Heart-Rate-Zone (HR = 134 bpm), which was signiﬁcantly higher than the average HR (103 bpm) from the comparison condition (t = 7.212, p < 0.001).",needs_review
3319,paper_194,"Impact of an active educational video game on children’s motivation, science knowledge, and physical activity",2014,not_reported,not_reported,"In this randomized controlled study, 53 elementary school students were assigned to an experimental condition or a comparison condition.",randomized_controlled_trial,science knowledge (test performance),Learning,NaN,η2 = 0.486 (pre-post),p < 0.001 (pre-post); between-group difference not significant,randomized_controlled_trial,0.2250,"students in both groups performed better on the post-test than they did on the pre-test (p < 0.001, η2 = 0.486), and their post-test scores did not differ signiﬁcantly.",needs_review
3299,paper_191,This is an open access article under the CC BY-NC-ND license (https://creativecommons.org/licenses/by-nc-nd/4.0) Peer-review under responsibility of the scientific committee of the 5th International Conference on Com...,2021,not_reported,not_reported,"Initial study: 297 participants split into 2 groups (149 played video games, 148 used traditional educational app). Follow-up study: 122 participants split into 2 groups (64 played video games, 58 used traditional ed...",randomized_controlled_trial,behaviour change in decision-making,interpreted_construct,positive,not_reported,not_reported,randomized_controlled_trial,0.2250,"Physicians who played the video game has more significant behaviour change. Therefore, the video game has bigger impact to participants compared to traditional educational app.",needs_review
463,paper_009,Intense acts of violence during video game play make daily life aggression appear innocuous: A new mechanism why violent video games increase aggression,2012,Penguin,not_reported,"Participants were 82 adults (61 women, 21 men, mean age: 21.9 years).",randomized_controlled_trial,game liking,Psychological,negative,d = 1.48,< .001,randomized_controlled_trial,0.2250,"Liking of the violent video game (M 

In [18]:
answer = llm_synthesize(
    "What behavioral outcomes are linked to video games or game mechanics in the evidence dataset?",
    behavioral,
    max_rows=25,
)

print(answer)

Supported findings
- Physician decision-making behaviour / behaviour change in decision-making — positive effect reported: physicians who played the video game showed greater behaviour change versus a traditional educational app (paper_191). (Effect size and p not reported; source reports measured decisions and recorded actions.)
- Physical activity intensity (heart rate) — increased during an active educational video game: experimental group HR = 134 bpm vs comparison 103 bpm, t = 7.212, p < 0.001 (paper_194).
- Video-game play time (minutes/day) — intervention (active game upgrade) group spent less time playing all video games over 12 weeks than controls: 54 vs 98 min/day (difference = −44 min/day, 95% CI [−92, 2]); p = 0.06 (trend toward decrease, paper_287).

Caveats
- The physician decision-making result (paper_191) lacks reported effect sizes, CIs, or p-values; details and robustness are unclear.
- The reduced play-time finding (paper_287) did not reach conventional significance 

In [19]:
aggression = search_evidence(
    term="aggression",
    require_supported=True,
    n=50,
)

display(summarize_counts(aggression))
display(aggression)

,rows,papers,games,mechanics,outcomes,positive,negative,mixed,null
0,50,8,10,1,18,22,10,2,0


,paper_id,paper_title,year,game_name,mechanic_name,population_raw,study_design,outcome_canonical,outcome_category,effect_direction,effect_size_raw,p_value,evidence_strength,edge_weight,source_quote,review_status
457,paper_009,Intense acts of violence during video game play make daily life aggression appear innocuous: A new mechanism why violent video games increase aggression,2012,Trooper Assassin,not_reported,"Participants were 82 adults (61 women, 21 men, mean age: 21.9 years).",randomized_controlled_trial,game liking,Psychological,negative,d = 1.48,< .001,randomized_controlled_trial,0.2250,"Liking of the violent video game (M = 2.32, SD = 1.27) was lower than liking of the neutral video game (M = 4.40, SD = 1.53), t(80) = 6.65, p b .001, d = 1.48.",needs_review
453,paper_009,Intense acts of violence during video game play make daily life aggression appear innocuous: A new mechanism why violent video games increase aggression,2012,Counterstrike,not_reported,"Participants were 82 adults (61 women, 21 men, mean age: 21.9 years).",randomized_controlled_trial,mood,Psychological,NaN,d = 0.39,0.081,randomized_controlled_trial,0.2250,"Playing the violent video game (M = 4.56, SD = 1.29) led to more negative mood scores than playing the neutral video game (M = 5.02, SD = 1.06), but the difference did not achieve traditional levels of statistical si...",needs_review
459,paper_009,Intense acts of violence during video game play make daily life aggression appear innocuous: A new mechanism why violent video games increase aggression,2012,Super Bubbles,not_reported,"Participants were 82 adults (61 women, 21 men, mean age: 21.9 years).",randomized_controlled_trial,mood,Psychological,NaN,d = 0.39,0.081,randomized_controlled_trial,0.2250,"Playing the violent video game (M = 4.56, SD = 1.29) led to more negative mood scores than playing the neutral video game (M = 5.02, SD = 1.06), but the difference did not achieve traditional levels of statistical si...",needs_review
458,paper_009,Intense acts of violence during video game play make daily life aggression appear innocuous: A new mechanism why violent video games increase aggression,2012,Super Bubbles,not_reported,"Participants were 82 adults (61 women, 21 men, mean age: 21.9 years).",randomized_controlled_trial,perceived aggressiveness,Psychological,negative,d = 0.84,< .001,randomized_controlled_trial,0.2250,"participants who played the violent video game perceived their daily life aggressive behavior as being less aggressive (M = 4.23, SD = 2.40; contrast weight: 3) compared to participants who played the violent video g...",needs_review
462,paper_009,Intense acts of violence during video game play make daily life aggression appear innocuous: A new mechanism why violent video games increase aggression,2012,Penguin,not_reported,"Participants were 82 adults (61 women, 21 men, mean age: 21.9 years).",randomized_controlled_trial,mood,Psychological,NaN,d = 0.39,0.081,randomized_controlled_trial,0.2250,"Playing the violent video game (M = 4.56, SD = 1.29) led to more negative mood scores than playing the neutral video game (M = 5.02, SD = 1.06), but the difference did not achieve traditional levels of statistical si...",needs_review
456,paper_009,Intense acts of violence during video game play make daily life aggression appear innocuous: A new mechanism why violent video games increase aggression,2012,Trooper Assassin,not_reported,"Participants were 82 adults (61 women, 21 men, mean age: 21.9 years).",randomized_controlled_trial,mood,Psychological,NaN,d = 0.39,0.081,randomized_controlled_trial,0.2250,"Playing the violent video game (M = 4.56, SD = 1.29) led to more negative mood scores than playing the neutral video game (M = 5.02, SD = 1.06), but the difference did not achieve traditional levels of statistical si...",needs_review
455,paper_009,Intense acts of violence during video game play make daily life aggression appear innocuous: A new mechanism why violent video games increase aggression,

In [20]:
answer = llm_synthesize(
    "What does the evidence say about video games and aggression-related behavioral outcomes?",
    aggression,
    max_rows=25,
)

print(answer)

Supported findings
- No behavioral effect in a lab experiment of violent vs nonviolent (and competitive) game versions: paper_129 (N=64; 2x2 design) reported that neither violence nor competition impacted aggressive behaviour (TCRTT). (paper_129)
- Short-term increase on a self-report aggression index after play: paper_181 (N=24, pupils aged 12–14) reported the Buss–Durkey aggression score rose from 17.3 before the game to 21.7 after the game. (paper_181)

Related psychological finding (not direct behavior)
- Players of violent games reported perceiving their own daily-life aggressive acts as less aggressive (perceived aggressiveness decreased; d = 0.84, p < .001) in paper_009 (N=82 adults). This is a subjective/perceptual outcome rather than an observed behavioral measure. (paper_009)

Caveats
- Measurement and reporting problems: paper_129 noted poor validity of the TCRTT and a delay between play and the aggression task; paper_129 and paper_181 did not report effect sizes/p-values fo

In [21]:
learning = supported(df)

learning = learning[
    learning[["outcome_category", "outcome_canonical", "outcome_raw", "paper_title", "source_quote"]]
    .astype(str)
    .agg(" ".join, axis=1)
    .str.contains(
        "education|learning|student|students|school|teacher|teachers|academic|achievement|motivation|knowledge|skills|vocabulary|STEM|geography|math|mathematics|literacy",
        case=False,
        na=False,
        regex=True,
    )
]

learning = learning[
    [col for col in ANSWER_COLS if col in learning.columns]
].sort_values("edge_weight", ascending=False)

display(summarize_counts(learning))
display(learning.head(50))

,rows,papers,games,mechanics,outcomes,positive,negative,mixed,null
0,1152,90,123,31,251,863,116,78,0


,paper_id,paper_title,year,game_name,mechanic_name,population_raw,study_design,outcome_canonical,outcome_category,effect_direction,effect_size_raw,p_value,evidence_strength,edge_weight,source_quote,review_status
1536,paper_071,Video games can develop graduate skills in higher education students: A randomised trial,2016,Minecraft,Building,Undergraduate students in the Arts and Humanities,randomized_controlled_trial,adaptability,graduate_attribute,positive,mean score change 1.15 standard deviations more positive in the intervention group than the control,0.002,randomized_controlled_trial,0.225,"A large effect size was observed with mean score change 1.1, 1.15, and 0.9 standard deviations more positive in the intervention group than the control on the communication, adaptability, and resourcefulness scales r...",needs_review
3170,paper_174,Does playing video games improve laparoscopic skills?,2012,NaN,not_reported,30 surgical novices were matched and randomised to 5 week VG training to in Half Life/HL (similar to endoscopy n = 15) or Chessmaster/CM (cognitive skills n = 15) and 10 in the control group with no game training.,randomized_controlled_trial,simulator performance (GI Mentor II surface examined),Performance,positive,HL: Diff = 4.60 (p = 0.039); CM: Diff = 1.67 (p = 0.20); Cont: Diff = 0.50 (p = 0.79),p = 0.039; p = 0.20; p = 0.79,randomized_controlled_trial,0.225,Only the HL group showed a signiﬁcant improvement on the “surface examined” test (p = 0.039). HL: Diff = 4.60 (p = 0.039) CM: Diff = 1.67 (p = 0.20) Cont: Diff = 0.50 (p = 0.79),needs_review
1564,paper_071,Video games can develop graduate skills in higher education students: A randomised trial,2016,Minecraft,Exploration,Undergraduate students in the Arts and Humanities,randomized_controlled_trial,adaptability,graduate_attribute,positive,mean score change 1.15 standard deviations more positive in the intervention group than the control,0.002,randomized_controlled_trial,0.225,"A large effect size was observed with mean score change 1.1, 1.15, and 0.9 standard deviations more positive in the intervention group than the control on the communication, adaptability, and resourcefulness scales r...",needs_review
1565,paper_071,Video games can develop graduate skills in higher education students: A randomised trial,2016,Minecraft,Perspective Taking,Undergraduate students in the Arts and Humanities,randomized_controlled_trial,adaptability,graduate_attribute,positive,mean score change 1.15 standard deviations more positive in the intervention group than the control,0.002,randomized_controlled_trial,0.225,"A large effect size was observed with mean score change 1.1, 1.15, and 0.9 standard deviations more positive in the intervention group than the control on the communication, adaptability, and resourcefulness scales r...",needs_review
1566,paper_071,Video games can develop graduate skills in higher education students: A randomised trial,2016,Minecraft,Cooperative Play,Undergraduate students in the Arts and Humanities,randomized_controlled_trial,resourcefulness,graduate_attribute,positive,mean score change 0.9 standard deviations more positive in the intervention group than the control,0.013,randomized_controlled_trial,0.225,"A large effect size was observed with mean score change 1.1, 1.15, and 0.9 standard deviations more positive in the intervention group than the control on the communication, adaptability, and resourcefulness scales r...",needs_review
1567,paper_071,Video games can develop graduate skills in higher education students: A randomised trial,2016,Minecraft,Building,Undergraduate students in the Arts and Humanities,randomized_controlled_trial,resourcefulness,graduate_attribute,positive,mean score change 0.9 standard deviations more positive in the intervention group than the control,0.013,randomized_controlled_trial,0.225,"A large effect size was observed with mean score change 1.1, 1.15, and 0.9 standard deviations more positive in the intervention group tha

In [22]:
answer = llm_synthesize(
    "What does the evidence say about games for education or learning outcomes?",
    learning,
    max_rows=25,
)

print(answer)

Supported findings
- paper_071: A randomized trial in undergraduate Arts/Humanities students found large, statistically significant improvements in graduate-attribute scores after game-based interventions—communication: +1.1 SD (p=0.004), adaptability: +1.15 SD (p=0.002), resourcefulness: +0.9 SD (p=0.013) (games: Minecraft, Borderlands 2 reported).  
- paper_069: A randomized trial with 83 adolescents (15–16 y) reported that commercial casual games (e.g., Cut the Rope, Angry Birds Space) "supported the elaboration potential of learned physical concepts" as measured by concept maps (effect direction: positive); effect sizes and p-values not reported.  
- paper_174: A randomized trial of 30 surgical novices showed that 5 weeks of Half-Life training improved laparoscopic simulator performance ("surface examined") (Diff = 4.60, p = 0.039); a Chessmaster group and the no‑training control did not show significant improvements.

Caveats
- Reporting gaps: paper_069 did not report effect sizes

In [25]:
stem_learning = learning[
    learning[["paper_title", "outcome_canonical",  "source_quote"]]
    .astype(str)
    .agg(" ".join, axis=1)
    .str.contains(
        "STEM|science|math|mathematics|geography|genetics|computational|vocabulary|spelling",
        case=False,
        na=False,
        regex=True,
    )
]

display(summarize_counts(stem_learning))
display(stem_learning.head(50))

,rows,papers,games,mechanics,outcomes,positive,negative,mixed,null
0,428,35,50,24,98,291,85,29,0


,paper_id,paper_title,year,game_name,mechanic_name,population_raw,study_design,outcome_canonical,outcome_category,effect_direction,effect_size_raw,p_value,evidence_strength,edge_weight,source_quote,review_status
3347,paper_196,Use of commercial video games to improve postural balance in patients with multiple sclerosis: A systematic review and meta-analysis of randomised controlled clinical trials,2021,NaN,not_reported,patients with multiple sclerosis (MS),randomized_controlled_trial,clinical_significance_relative_to_minimum_detectable_change,postural_balance_interpretation,NaN,not greater than the minimum detectable change reported in the literature,not_reported,randomized_controlled_trial,0.2250,but these were not greater than the minimum detectable change reported in the literature.,needs_review
3346,paper_196,Use of commercial video games to improve postural balance in patients with multiple sclerosis: A systematic review and meta-analysis of randomised controlled clinical trials,2021,NaN,not_reported,patients with multiple sclerosis (MS),randomized_controlled_trial,Berg Balance Scale (BBS),postural_balance,positive,DM: 5.30,< .001,randomized_controlled_trial,0.2250,"We did observe intergroup differences in BBS scores in favour of video game therapy (DM: 5.30; 95% CI,3.39—7.21; P < .001; I2 = 0%), but these were not greater than the minimum detectable change reported in the liter...",needs_review
3345,paper_196,Use of commercial video games to improve postural balance in patients with multiple sclerosis: A systematic review and meta-analysis of randomised controlled clinical trials,2021,NaN,not_reported,patients with multiple sclerosis (MS),randomized_controlled_trial,Timed 25-Foot Walk,postural_balance,NaN,DM: .15,.75,randomized_controlled_trial,0.2250,"We found no signiﬁcant differences between the video game therapy group and the control group in ... Timed 25-Foot Walk scores (DM: .15; 95% CI, −1.06 to .76; P = .75; I2 = 0%)",needs_review
537,paper_015,Can video games enhance surgical skills acquisition for medical students? A systematic review,2020,NaN,not_reported,40 students,randomized_controlled_trial,time_to_completion,surgical_task_performance,positive,71.7 to 44.0 s vs 50.8 to 39.7 s,P = .006,randomized_controlled_trial,0.2250,"Harrington et al,14 who randomized 40 students to either a Wii U VGBT group or a non-VGBT control group, who then performed LAP Mentor tasks. Signiﬁcant reductions in time to completion were observed (71.7 to 44.0 s ...",needs_review
536,paper_015,Can video games enhance surgical skills acquisition for medical students? A systematic review,2020,Wii U,not_reported,40 students,randomized_controlled_trial,time_to_completion,surgical_task_performance,positive,71.7 to 44.0 s vs 50.8 to 39.7 s,P = .006,randomized_controlled_trial,0.2250,"Harrington et al,14 who randomized 40 students to either a Wii U VGBT group or a non-VGBT control group, who then performed LAP Mentor tasks. Signiﬁcant reductions in time to completion were observed (71.7 to 44.0 s ...",needs_review
3344,paper_196,Use of commercial video games to improve postural balance in patients with multiple sclerosis: A systematic review and meta-analysis of randomised controlled clinical trials,2021,NaN,not_reported,patients with multiple sclerosis (MS),randomized_controlled_trial,Four Step Square Test,postural_balance,NaN,DM: −.74,.48,randomized_controlled_trial,0.2250,"We found no signiﬁcant differences between the video game therapy group and the control group in Four Step Square Test (DM: −.74; 95% CI, −2.79 to 1.32; P = .48; I2 = 0%)",needs_review
2644,paper_158,Digital game-based spelling intervention for children with spelling deficits: A randomized controlled trial,2024,NaN,not_reported,Participants were 65 German speaking second- and third-graders with spelling disorder.,randomized_controlled_trial,phoneme-grapheme mapping,precursor_skill,positive,not_reported,not_reported,randomized_controlled_trial,0.2250,"Results showed inter

In [26]:
answer = llm_synthesize(
    "What evidence links games to STEM, school, or academic learning outcomes?",
    stem_learning,
    max_rows=25,
)

print(answer)

Supported findings (papers and key results)
- paper_194 — Elementary school RCT (n=53): both game and comparison groups improved on a science knowledge test pre→post (p < .001), and the experimental group reported higher situational interest (p < .01, η2 = 0.301). Between-group post-test scores did not differ significantly.  
- paper_158 — RCT with 65 German 2nd–3rd graders with spelling disorder: the digital game intervention produced positive effects on precursor literacy skills (phoneme–grapheme mapping and phoneme awareness). (Results reported as positive; effect sizes/p-values not provided.)  
- paper_182 — RCT with 24 low‑intermediate EFL learners: the experimental (game) group outperformed the control group on an English vocabulary post-test (statistically significant).  
- paper_191 — RCTs (adults/physicians and general participants): playing video games produced greater behaviour change in clinical decision‑making versus a traditional educational app, and action‑game training 

In [27]:
tech_econ = supported(df)

tech_econ = tech_econ[
    tech_econ[["outcome_category", "outcome_canonical", "outcome_raw", "paper_title", "source_quote"]]
    .astype(str)
    .agg(" ".join, axis=1)
    .str.contains(
        "technology|technological|innovation|business model|market|economic|economics|purchase|intention|revenue|sales|monetization|microtransaction|freemium|subscription|consumer|brand|product|platform|console|mobile gaming|industry|user experience|usage|loyalty",
        case=False,
        na=False,
        regex=True,
    )
]

tech_econ = tech_econ[
    [col for col in ANSWER_COLS if col in tech_econ.columns]
].sort_values("edge_weight", ascending=False)

display(summarize_counts(tech_econ))
display(tech_econ.head(50))

,rows,papers,games,mechanics,outcomes,positive,negative,mixed,null
0,320,43,60,26,96,235,50,23,0


,paper_id,paper_title,year,game_name,mechanic_name,population_raw,study_design,outcome_canonical,outcome_category,effect_direction,effect_size_raw,p_value,evidence_strength,edge_weight,source_quote,review_status
2526,paper_145,"To screen, or not to screen: An experimental comparison of two methods for correlating video game loot box expenditure and problem gambling severity",2024,loot boxes,not_reported,N = 2027; non-screening approach; n = 1005; screening participants; n = 1022,randomized_controlled_trial,Problem gambling severity (PGSI) scores were inflated in the non-screening group relative to the screening group,interpreted_construct,positive,not_reported,not_reported,randomized_controlled_trial,0.2250,"However, problem gambling severity was inflated in the non-screening group, meaning that future research on gambling-like products should include gambling participation screening questions; better define what ‘gambli...",needs_review
2523,paper_145,"To screen, or not to screen: An experimental comparison of two methods for correlating video game loot box expenditure and problem gambling severity",2024,loot boxes,not_reported,N = 2027; non-screening approach; n = 1005; screening participants; n = 1022,randomized_controlled_trial,Problem gambling severity (PGSI) scores were inflated in the non-screening group relative to the screening group,interpreted_construct,positive,not_reported,not_reported,randomized_controlled_trial,0.2250,"However, problem gambling severity was inflated in the non-screening group, meaning that future research on gambling-like products should include gambling participation screening questions; better define what ‘gambli...",needs_review
4940,paper_296,Open Access Game-based learning and gamification in initial teacher training in the social sciences: an experiment with MinecraftEdu,2012,MinecraftEdu,Building,second-year university students who are pursuing a degree in Primary Education and are enrolled in a subject entitled Social Sciences II: History and Teaching at Castilla-La Mancha University.,quasi_experimental,educational innovation,learning/innovation,positive,not_reported,not_reported,quasi_experimental,0.1687,"Interest level, educational innovation and motivation are valued positively and show statistically significant improvements.",needs_review
4938,paper_296,Open Access Game-based learning and gamification in initial teacher training in the social sciences: an experiment with MinecraftEdu,2012,MinecraftEdu,Perspective Taking,second-year university students who are pursuing a degree in Primary Education and are enrolled in a subject entitled Social Sciences II: History and Teaching at Castilla-La Mancha University.,quasi_experimental,interest,engagement/motivation,positive,not_reported,not_reported,quasi_experimental,0.1687,"Interest level, educational innovation and motivation are valued positively and show statistically significant improvements.",needs_review
4937,paper_296,Open Access Game-based learning and gamification in initial teacher training in the social sciences: an experiment with MinecraftEdu,2012,MinecraftEdu,Exploration,second-year university students who are pursuing a degree in Primary Education and are enrolled in a subject entitled Social Sciences II: History and Teaching at Castilla-La Mancha University.,quasi_experimental,interest,engagement/motivation,positive,not_reported,not_reported,quasi_experimental,0.1687,"Interest level, educational innovation and motivation are valued positively and show statistically significant improvements.",needs_review
4936,paper_296,Open Access Game-based learning and gamification in initial teacher training in the social sciences: an experiment with MinecraftEdu,2012,MinecraftEdu,Building,second-year university students who are pursuing a degree in Primary Education and are enrolled in a subject entitled Social Sciences II: History and Teaching at Castilla-La Mancha University.,quasi_experimental,interest,engagement/motivation,positive

In [28]:
answer = llm_synthesize(
    "What does the evidence say about technological or economic impacts of games and game platforms?",
    tech_econ,
    max_rows=25,
)

print(answer)

Supported findings
- Economic (substitution) effects — paper_255: Analysis of ~100,000 panel observations in the Japanese video‑game market found that play on smart devices/mobile games produced recognizable negative substitution effects on handheld/older consoles (3DS, Wii, PSP) but not on PS3; the substitution magnitudes were small (at most about 0.1). (paper_255)
- Technological/affordance effects — paper_080: In a quasi‑experimental study (N=190) of a serious game virtual environment, higher perceived contingency (an interactivity affordance) was associated with greater self‑efficacy. (paper_080)
- Technological/educational uptake — paper_296: Use of MinecraftEdu in initial teacher training was associated with increased interest, educational innovation and motivation (statistically significant improvements reported). (paper_296)

Caveats
- Most entries are quasi‑experimental; randomized details and inferential statistics (effect sizes, p‑values) are generally not reported in these 

In [29]:
monetization = supported(df)

monetization = monetization[
    monetization[["paper_title", "outcome_canonical", "outcome_raw", "source_quote"]]
    .astype(str)
    .agg(" ".join, axis=1)
    .str.contains(
        "purchase|intention|microtransaction|freemium|subscription|revenue|sales|monetization|loot box|consumer|brand|loyalty|business model|market",
        case=False,
        na=False,
        regex=True,
    )
]

monetization = monetization[
    [col for col in ANSWER_COLS if col in monetization.columns]
].sort_values("edge_weight", ascending=False)

display(summarize_counts(monetization))
display(monetization.head(50))

,rows,papers,games,mechanics,outcomes,positive,negative,mixed,null
0,162,23,20,4,56,115,26,15,0


,paper_id,paper_title,year,game_name,mechanic_name,population_raw,study_design,outcome_canonical,outcome_category,effect_direction,effect_size_raw,p_value,evidence_strength,edge_weight,source_quote,review_status
2521,paper_145,"To screen, or not to screen: An experimental comparison of two methods for correlating video game loot box expenditure and problem gambling severity",2024,loot boxes,not_reported,N = 2027; non-screening approach; n = 1005; screening participants; n = 1022,randomized_controlled_trial,Loot box expenditure correlated with problem gambling severity (PGSI),game_level,positive,not_reported,not_reported,randomized_controlled_trial,0.2250,Results showed positive correlations between loot box spending and problem gambling across both experimental conditions.,needs_review
2522,paper_145,"To screen, or not to screen: An experimental comparison of two methods for correlating video game loot box expenditure and problem gambling severity",2024,loot boxes,not_reported,N = 2027; non-screening approach; n = 1005; screening participants; n = 1022,randomized_controlled_trial,Past-year gambling participation (binary) positively correlated with loot box expenditure (in screening group),game_level,positive,not_reported,not_reported,randomized_controlled_trial,0.2250,"In addition, a predicted positive correlation emerged between binary past-year gambling participation and loot box expenditure in the screening group.",needs_review
2523,paper_145,"To screen, or not to screen: An experimental comparison of two methods for correlating video game loot box expenditure and problem gambling severity",2024,loot boxes,not_reported,N = 2027; non-screening approach; n = 1005; screening participants; n = 1022,randomized_controlled_trial,Problem gambling severity (PGSI) scores were inflated in the non-screening group relative to the screening group,interpreted_construct,positive,not_reported,not_reported,randomized_controlled_trial,0.2250,"However, problem gambling severity was inflated in the non-screening group, meaning that future research on gambling-like products should include gambling participation screening questions; better define what ‘gambli...",needs_review
2524,paper_145,"To screen, or not to screen: An experimental comparison of two methods for correlating video game loot box expenditure and problem gambling severity",2024,loot boxes,not_reported,N = 2027; non-screening approach; n = 1005; screening participants; n = 1022,randomized_controlled_trial,Loot box expenditure correlated with problem gambling severity (PGSI),game_level,positive,not_reported,not_reported,randomized_controlled_trial,0.2250,Results showed positive correlations between loot box spending and problem gambling across both experimental conditions.,needs_review
2525,paper_145,"To screen, or not to screen: An experimental comparison of two methods for correlating video game loot box expenditure and problem gambling severity",2024,loot boxes,not_reported,N = 2027; non-screening approach; n = 1005; screening participants; n = 1022,randomized_controlled_trial,Past-year gambling participation (binary) positively correlated with loot box expenditure (in screening group),game_level,positive,not_reported,not_reported,randomized_controlled_trial,0.2250,"In addition, a predicted positive correlation emerged between binary past-year gambling participation and loot box expenditure in the screening group.",needs_review
2526,paper_145,"To screen, or not to screen: An experimental comparison of two methods for correlating video game loot box expenditure and problem gambling severity",2024,loot boxes,not_reported,N = 2027; non-screening approach; n = 1005; screening participants; n = 1022,randomized_controlled_trial,Problem gambling severity (PGSI) scores were inflated in the non-screening group relative to the screening group,interpreted_construct,positive,not_reported,not_reported,randomized_controlled_trial,0.2250,"However, problem gambling severity was inflated in the n

In [30]:
answer = llm_synthesize(
    "What evidence links game design, monetization, or platforms to economic outcomes?",
    monetization,
    max_rows=25,
)

print(answer)

Supported findings
- Loot box monetization: Higher loot box expenditure is positively associated with problem gambling severity across both experimental conditions (paper_145). The same dataset also found past‑year gambling participation (binary) positively correlated with loot box spending in the screening group (paper_145).
- Subscription platform (Xbox Game Pass / Game Pass): Perceived value factors (playing-games value, enjoyment, and price) positively contribute to continual use intention for the subscription service; variety preference and FoMO also positively contribute to perceived value (paper_147).

Caveats
- Effect sizes, p-values, and detailed statistics are not reported in these rows (paper_145, paper_147). 
- Loot box findings note measurement sensitivity: PGSI scores were inflated in the non‑screening group, indicating screening/definition issues that may affect estimates (paper_145).
- Game Pass evidence is correlational and based on self‑reported perceived values and i

In [31]:
health_terms = (
    "physiological|health|physical activity|exercise|energy expenditure|MET|heart rate|"
    "balance|motor|mobility|oxygen|ventilation|postural|fitness|BMI|body composition|"
    "rehabilitation|Parkinson|multiple sclerosis|cystic fibrosis|cancer|stroke|older adults|"
    "scoliosis|traumatic brain injury|pain|anxiety|sleep"
)

health = supported(df)

health = health[
    health[["paper_title", "population_raw", "outcome_category", "outcome_canonical", "outcome_raw", "measurement_raw", "source_quote"]]
    .astype(str)
    .agg(" ".join, axis=1)
    .str.contains(health_terms, case=False, na=False, regex=True)
]

health = health[
    [col for col in ANSWER_COLS if col in health.columns]
].sort_values("edge_weight", ascending=False)

display(summarize_counts(health))
display(health.head(75))

,rows,papers,games,mechanics,outcomes,positive,negative,mixed,null
0,1634,106,169,32,304,947,154,65,0


,paper_id,paper_title,year,game_name,mechanic_name,population_raw,study_design,outcome_canonical,outcome_category,effect_direction,effect_size_raw,p_value,evidence_strength,edge_weight,source_quote,review_status
4734,paper_286,Open Access Computer game-based and traditional learning method: a comparison regarding students’ knowledge retention,2013,NaN,not_reported,Speech-Language and Hearing pathology undergraduate students.,randomized_controlled_trial,long-term Physiology knowledge (6 months post-test),Knowledge retention,negative,not_reported,not_reported,randomized_controlled_trial,0.225,Students that received the traditional lecture performed better in both post-test and long-term post-test when considering the Anatomy and Physiology questions.,needs_review
4732,paper_286,Open Access Computer game-based and traditional learning method: a comparison regarding students’ knowledge retention,2013,NaN,not_reported,Speech-Language and Hearing pathology undergraduate students.,randomized_controlled_trial,short-term Physiology knowledge (post-test),Knowledge retention,negative,not_reported,not_reported,randomized_controlled_trial,0.225,Students that received the traditional lecture performed better in both post-test and long-term post-test when considering the Anatomy and Physiology questions.,needs_review
4751,paper_287,Page 1 of 5 (page number not for citation purposes),2008,NaN,not_reported,Twenty children (mean ± SD age = 12 ± 1.5 years; 40% female) were randomised to receive either an active video game upgrade package or to a control group (no intervention).,randomized_controlled_trial,physical activity (accelerometer counts/min) at 6 weeks,physical_activity_objective,positive,difference at 6 weeks = 194 counts/min,0.04,randomized_controlled_trial,0.225,"Average time spent in all physical activities measured with an accelerometer was higher in the active video game intervention group compared to the control group (difference at 6 weeks = 194 counts/min, p = 0.04...",needs_review
4749,paper_287,Page 1 of 5 (page number not for citation purposes),2008,PlayStation®2,not_reported,Twenty children (mean ± SD age = 12 ± 1.5 years; 40% female) were randomised to receive either an active video game upgrade package or to a control group (no intervention).,randomized_controlled_trial,physical activity (accelerometer counts/min) at 12 weeks,physical_activity_objective,NaN,difference at 12 weeks = 48 counts/min,0.06,randomized_controlled_trial,0.225,"...and at 12 weeks = 48 counts/min, p = 0.06).",needs_review
4748,paper_287,Page 1 of 5 (page number not for citation purposes),2008,PlayStation®2,not_reported,Twenty children (mean ± SD age = 12 ± 1.5 years; 40% female) were randomised to receive either an active video game upgrade package or to a control group (no intervention).,randomized_controlled_trial,physical activity (accelerometer counts/min) at 6 weeks,physical_activity_objective,positive,difference at 6 weeks = 194 counts/min,0.04,randomized_controlled_trial,0.225,"Average time spent in all physical activities measured with an accelerometer was higher in the active video game intervention group compared to the control group (difference at 6 weeks = 194 counts/min, p = 0.04...",needs_review
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,paper_257,Open Access The effects of video game therapy on balance and attention in chronic ambulatory traumatic brain injury: an exploratory study,2013,X-Box 360 Kinect,Gesture Input,We enrolled chronic traumatic brain injury patients (n = 21) that randomly received VGT or BPT for 3 sessions per week for 6 weeks.,randomized_controlled_trial,Community Balance and Mobility Scale (CB&M),Balance/Mobility,positive,not_reported,not_reported,randomized_controlled_trial,0.225,"Both groups improved in CB&M scores, but only the VGT group increased on the UBS and TUG with a between-group significance (p < 0.05). Selective attention improved significantly in the VGT group (p < 0.01)."

In [33]:
impact_patterns = {
    "energy_expenditure_activity": "energy expenditure|MET|physical activity|activity counts|exercise intensity|moderate-to-vigorous|MVPA",
    "cardiorespiratory": "heart rate|oxygen|SpO2|ventilation|VO2|cardiorespiratory|aerobic",
    "balance_motor_rehab": "balance|postural|motor|mobility|rehabilitation|Parkinson|stroke|traumatic brain injury|scoliosis",
    "body_composition_fitness": "BMI|body composition|weight|fitness|aerobic fitness|excess weight|obesity",
    "pain_anxiety_clinical": "pain|anxiety|procedural|cancer|cystic fibrosis|multiple sclerosis|clinical",
    "sleep_fatigue": "sleep|fatigue",
}

health_impact_rows = []

for impact, pattern in impact_patterns.items():
    subset = health[
        health[["paper_title", "population_raw", "outcome_category", "outcome_canonical",  "source_quote"]]
        .astype(str)
        .agg(" ".join, axis=1)
        .str.contains(pattern, case=False, na=False, regex=True)
    ]
    
    health_impact_rows.append({
        "impact_area": impact,
        "rows": len(subset),
        "papers": subset["paper_id"].nunique(),
        "positive": int((subset["effect_direction"] == "positive").sum()),
        "negative": int((subset["effect_direction"] == "negative").sum()),
        "mixed": int((subset["effect_direction"] == "mixed").sum()),
        "null": int((subset["effect_direction"] == "null").sum()),
        "top_games": ", ".join(subset["game_name"].dropna().value_counts().head(5).index.astype(str)),
        "top_outcomes": ", ".join(subset["outcome_canonical"].dropna().value_counts().head(5).index.astype(str)),
    })

health_impact_summary = pd.DataFrame(health_impact_rows).sort_values("papers", ascending=False)
display(health_impact_summary)

,impact_area,rows,papers,positive,negative,mixed,null,top_games,top_outcomes
0,energy_expenditure_activity,758,43,417,64,0,0,"custom-made action video game, Wii boxing, Wii Fit Plus, virtual reality, Your Shape Fitness Evolved (Tai Chi)","perceived exertion (RPE Borg scale), activity counts (accelerometer-derived), energy expenditure (METs; kJ/min), correlation(energy expenditure/activity counts, balance status), information processing (overall)"
4,pain_anxiety_clinical,151,18,61,60,0,0,"virtual reality, conventional video game, distracting games, interactive audiovisual game, preparatory games","anxiety, procedural pain, acceptability/satisfaction, adverse effects, need for physical restraint during procedure"
2,balance_motor_rehab,306,17,187,26,1,0,"League of Legends, Kinect Sports, Kinect Adventures, X-Box 360 Kinect, Counter-Strike","correlation(energy expenditure/activity counts, balance status), reaction time (motor response speed), cancellation ability (stop-signal process), FPS viewing increases Fraction of Time of sensorimotor-related CAPs r..."
1,cardiorespiratory,218,11,98,23,37,0,"Wii Fit Plus, Kinect, first-person shooter, Wii Fit, EA SPORTS™ACTIVE","heart rate (HR), minute ventilation (VE), peak HR, VO2, VE (CPET), oxygen consumption (VO2), dyspnea"
3,body_composition_fitness,215,6,67,16,22,0,"Kinect, Wii Fit Plus, EA SPORTS™ACTIVE, Microsoft Xbox, Internet video games","energy cost (between-group comparison), minute ventilation (VE), dyspnea, oxygen uptake (VO2), Wii-Acti and Wii-Train impose metabolic demand comparable to 6MWT"
5,sleep_fatigue,40,4,12,24,0,0,"Nintendo Wii, Nintendo WiiFit™, World of Warcraft, MMORPG","daytime functioning, Irritability, sleep disturbance and mood changes (self-reported), sleep onset latency (delay), sleep continuity / stability / organization, sleep quality"


In [35]:
def show_health_impact(impact_area: str, n: int = 30):
    pattern = impact_patterns[impact_area]
    subset = health[
        health[["paper_title", "population_raw", "outcome_category", "outcome_canonical", "source_quote"]]
        .astype(str)
        .agg(" ".join, axis=1)
        .str.contains(pattern, case=False, na=False, regex=True)
    ]
    subset = subset.sort_values("edge_weight", ascending=False)
    display(summarize_counts(subset))
    display(subset.head(n))
    return subset

balance_rehab = show_health_impact("balance_motor_rehab")

,rows,papers,games,mechanics,outcomes,positive,negative,mixed,null
0,306,17,54,32,45,187,26,1,0


,paper_id,paper_title,year,game_name,mechanic_name,population_raw,study_design,outcome_canonical,outcome_category,effect_direction,effect_size_raw,p_value,evidence_strength,edge_weight,source_quote,review_status
4041,paper_257,Open Access The effects of video game therapy on balance and attention in chronic ambulatory traumatic brain injury: an exploratory study,2013,Biodex balance platform,not_reported,We enrolled chronic traumatic brain injury patients (n = 21) that randomly received VGT or BPT for 3 sessions per week for 6 weeks.,randomized_controlled_trial,Unified Balance Scale (UBS),Balance/Mobility,positive,not_reported,<0.05,randomized_controlled_trial,0.225,"Both groups improved in CB&M scores, but only the VGT group increased on the UBS and TUG with a between-group significance (p < 0.05).",needs_review
3999,paper_257,Open Access The effects of video game therapy on balance and attention in chronic ambulatory traumatic brain injury: an exploratory study,2013,X-Box 360 Kinect,Sports Simulation,We enrolled chronic traumatic brain injury patients (n = 21) that randomly received VGT or BPT for 3 sessions per week for 6 weeks.,randomized_controlled_trial,Unified Balance Scale (UBS),Balance/Mobility,positive,not_reported,<0.05,randomized_controlled_trial,0.225,"Both groups improved in CB&M scores, but only the VGT group increased on the UBS and TUG with a between-group significance (p < 0.05).",needs_review
4038,paper_257,Open Access The effects of video game therapy on balance and attention in chronic ambulatory traumatic brain injury: an exploratory study,2013,Kinect Sports,Sports Simulation,We enrolled chronic traumatic brain injury patients (n = 21) that randomly received VGT or BPT for 3 sessions per week for 6 weeks.,randomized_controlled_trial,Selective visual attention (Go/No-go task),Attention / Neurocognitive,positive,not_reported,<0.01,randomized_controlled_trial,0.225,Selective attention improved significantly in the VGT group (p < 0.01).,needs_review
4037,paper_257,Open Access The effects of video game therapy on balance and attention in chronic ambulatory traumatic brain injury: an exploratory study,2013,Kinect Sports,Gesture Input,We enrolled chronic traumatic brain injury patients (n = 21) that randomly received VGT or BPT for 3 sessions per week for 6 weeks.,randomized_controlled_trial,Selective visual attention (Go/No-go task),Attention / Neurocognitive,positive,not_reported,<0.01,randomized_controlled_trial,0.225,Selective attention improved significantly in the VGT group (p < 0.01).,needs_review
4033,paper_257,Open Access The effects of video game therapy on balance and attention in chronic ambulatory traumatic brain injury: an exploratory study,2013,Kinect Sports,Movement Control,We enrolled chronic traumatic brain injury patients (n = 21) that randomly received VGT or BPT for 3 sessions per week for 6 weeks.,randomized_controlled_trial,Timed Up and Go (TUG),Mobility/Functional mobility,positive,not_reported,<0.05,randomized_controlled_trial,0.225,"Both groups improved in CB&M scores, but only the VGT group increased on the UBS and TUG with a between-group significance (p < 0.05).",needs_review
4032,paper_257,Open Access The effects of video game therapy on balance and attention in chronic ambulatory traumatic brain injury: an exploratory study,2013,Kinect Sports,Sports Simulation,We enrolled chronic traumatic brain injury patients (n = 21) that randomly received VGT or BPT for 3 sessions per week for 6 weeks.,randomized_controlled_trial,Timed Up and Go (TUG),Mobility/Functional mobility,positive,not_reported,<0.05,randomized_controlled_trial,0.225,"Both groups improved in CB&M scores, but only the VGT group increased on the UBS and TUG with a between-group significance (p < 0.05).",needs_review
4031,paper_257,Open Access The effects of video game therapy on balance and attention in chronic ambulatory traumatic brain injury: an exploratory study,2013,Kinect Sports,Gesture Input,We enrolled chronic traum

In [36]:
answer = llm_synthesize(
    "What physiological and health impacts of games are supported by the evidence? Organize the answer by impact area.",
    health,
    max_rows=35,
)

print(answer)

Supported findings

- Physical activity (objective)
  - Active (motion/“exergame”) play increased accelerometer-measured activity at 6 weeks in children: difference = +194 counts/min, p = 0.04 (paper_287).
  - That effect attenuated by 12 weeks (difference = +48 counts/min, p = 0.06) (paper_287).

- Body composition / BMI
  - In overweight/obese children, an EyeToy exergame intervention produced small but statistically significant reductions at 24 weeks: BMI difference ≈ −0.33 (SE 0.12), p = 0.01; percent body fat difference ≈ −0.83 (SE 0.36), p = 0.02. Aerobic fitness partly mediated these effects (paper_288).

- Balance and mobility
  - Video-game-based therapy (VGT) improved balance/mobility measures vs comparator in chronic ambulatory traumatic brain injury: between-group improvement on Unified Balance Scale and Timed Up & Go, p < 0.05 (paper_257).

- Attention / Neurocognitive
  - Selective visual attention improved significantly after VGT (Go/No‑go task), p < 0.01 (paper_257).

-

In [37]:
answer = llm_synthesize(
    "What evidence links active video games or exergames to balance, motor, or rehabilitation outcomes?",
    balance_rehab,
    max_rows=25,
)

print(answer)

Supported findings (from paper_257)
- A randomized controlled trial in chronic ambulatory traumatic brain injury patients (n = 21) found that video-game therapy (VGT) produced between-group improvements versus conventional balance therapy on Unified Balance Scale (UBS) and Timed Up and Go (TUG) (p < 0.05). Games/platforms reported: X-Box 360 Kinect / Kinect Sports / Kinect Adventures / Biodex balance platform (paper_257).
- Community Balance and Mobility Scale (CB&M) scores improved in both groups; selective visual attention (Go/No‑go) improved significantly in the VGT group (p < 0.01) (paper_257).

Caveats
- Small sample (n = 21) and effect sizes not reported; many CB&M entries lack p-values (paper_257).
- Multiple games/mechanics were used (gesture/movement control, sports simulation); the specific contribution of any single game is unclear (paper_257).
- Population limited to chronic ambulatory TBI patients—generalizability to other groups is unknown (paper_257).
- Study labeled exp

In [38]:
answer = llm_synthesize(
    "What video game mechanics help in balance, motor, or rehabilitation outcomes?",
    balance_rehab,
    max_rows=25,
)

print(answer)

Supported findings (all from paper_id: paper_257)
- Movement Control (Kinect Sports / Kinect Adventures / X‑Box 360 Kinect) — associated with improved balance/mobility: increases on Unified Balance Scale (UBS, between‑group p < 0.05) and faster Timed Up and Go (TUG, between‑group p < 0.05). Also linked to improved Community Balance & Mobility (CB&M) (improvement reported).  
- Gesture Input (Kinect titles) — associated with improved balance/mobility (UBS, TUG) and with better selective visual attention on a Go/No‑go task (p < 0.01).  
- Sports Simulation (Kinect titles / X‑Box) — associated with improved balance/mobility (UBS, TUG, CB&M) and improved selective attention (p < 0.01 reported for attention).

Caveats
- All evidence comes from one small RCT (n = 21) in chronic ambulatory traumatic brain injury (paper_257).  
- Effect sizes were not reported; some outcomes (CB&M) improved in both groups, so not all improvements were uniquely attributable to video‑game therapy.  
- Several en